# MISO Benchmark for spatial mutliomcis data integration on simulated dataset

Notebook benchmarks spatial mutliomcis data integration using MISO on simulated dataset.

## Loading

In [ ]:
from miso.hist_features import get_features
from miso.utils import *
from miso import Miso
from PIL import Image
import pandas as pd
import numpy as np
import scanpy as sc
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
Image.MAX_IMAGE_PIXELS = None
Image.MAX_IMAGE_PIXELS = None
import torch
import random

seed=100
np.random.seed(seed)
torch.manual_seed(seed)
random.seed(seed)

if torch.cuda.is_available():
    device = 'cuda'
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print("CUDA is available. GPU:", torch.cuda.get_device_name(0))
else:
    device = 'cpu'
    print("CUDA is not available. Using CPU.")

In [ ]:
from matplotlib import rcParams
rcParams["figure.dpi"] = 300
rcParams["savefig.dpi"] = 300
rcParams["savefig.transparent"] = True
rcParams["figure.facecolor"] = 'white'
rcParams["axes.facecolor"] = 'white'

## MISO pipeline

In [ ]:
# Set the directory for the datasets and the output directory
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for i in range(1, 6):
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad data.")
    # Read the RNA and ATAC datasets
    adata_omics1 = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_omics2 = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_atac.h5ad')
    adata_omics2.obsm['spatial'] = adata_omics1.obsm['spatial']
    adata_omics1.var_names_make_unique()
    adata_omics2.var_names_make_unique()

    # Preprocess RNA data
    adata_omics1 = preprocess(adata_omics1, modality='rna')
    # Preprocess ATAC data
    adata_omics2 =preprocess(adata_omics2, modality='atac')

    # Define and train the model
    model = Miso([adata_omics1,adata_omics2],ind_views='all',combs='all',sparse=False,device=device)
    model.train()
    np.save('emb.npy', model.emb)
    # Copy the results to the RNA dataset
    adata = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata.obsm['MISO'] = model.emb

    # Perform clustering
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=adata.obsm['MISO'].shape[1],
                    use_rep='MISO')
    sc.tl.leiden(adata, resolution=0.6)

    # Plot the spatial clustering results
    sc.pl.spatial(adata, color=['cell_type', 'leiden'], spot_size=0.12, wspace=0.2)

    # Save the processed dataset
    adata.write_h5ad(f'{output_dir}/Simulated_Dataset_{i}/miso_multiomics.h5ad', compression='gzip')

In [ ]:
!pip list